[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gauravs19/iiot-predictive-maintenance/blob/main/notebooks/02_predictive_maintenance.ipynb)

# 02 · Predictive Maintenance

Two complementary models, two flavours of the same business goal — *act before the
machine breaks*:

1. **Failure classification** on **AI4I** — given one snapshot of a machine, will it
   fail? (supervised, tabular, tree model + SHAP explainability)
2. **RUL regression** on **C-MAPSS** — given an engine's recent sensor history, how
   many cycles of life remain? (supervised, sequence model / LSTM)

We explain every metric as we go, because *which* metric you optimise is the most
important modelling decision in PdM.

## Step 1 · Bootstrap + imports

In [ ]:
# --- Environment bootstrap (works locally AND on Google Colab) ---------------
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # On Colab there is no repo yet, so clone it and install dependencies.
    !git clone -q https://github.com/gauravs19/iiot-predictive-maintenance.git
    %cd iiot-predictive-maintenance
    !pip install -q -r requirements.txt

# Make the repo root importable so `from src import ...` works from notebooks/.
def _find_repo_root(start="."):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "src")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError("repo root (folder containing src/) not found")

REPO_ROOT = _find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print("running on Colab" if IN_COLAB else "running locally")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, precision_recall_curve)

from src import data, features, models, utils
print("imports OK")

## Part A — Failure classification (AI4I)

### Step 2 · Prepare features and split

`features.prepare_ai4i()` returns `(X, y)`:
- It **drops leakage columns** — the five individual failure-mode flags (TWF, HDF,
  …) literally encode the answer, so keeping them would be cheating.
- It one-hot-encodes the machine quality `Type` (L/M/H).

We then split into train/test with **stratification** so the rare failures appear in
the same proportion in both halves.

In [ ]:
ai4i = data.load_ai4i()
X, y = features.prepare_ai4i(ai4i)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("train:", X_train.shape, " test:", X_test.shape)
print("test failure rate: %.2f%%" % (100 * y_test.mean()))

### Step 3 · Train a Random Forest

We start with a **Random Forest** — an ensemble of decision trees. It's a strong,
low-fuss baseline for tabular data: handles non-linear interactions, needs little
tuning, and gives feature importances for free.

**`class_weight="balanced"`** is the key setting here: it tells the model to pay
proportionally more attention to the rare failure class, counteracting the 96/4
imbalance we saw in notebook `00`.

In [ ]:
clf = RandomForestClassifier(
    n_estimators=300, max_depth=None, class_weight="balanced",
    random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
print("trained Random Forest with", clf.n_estimators, "trees")

### Step 4 · Evaluate — and why accuracy is the wrong headline

We look at three things:

- **Confusion matrix** — counts of true/false positives/negatives. In PdM the
  expensive mistake is a **false negative** (a real failure we missed).
- **Precision / recall / F1** — *recall* on the failure class = "of all real
  failures, what fraction did we catch?"; *precision* = "of all failure alarms, what
  fraction were real?". These matter far more than overall accuracy.
- **ROC-AUC** — threshold-independent ranking quality (1.0 = perfect, 0.5 = random).

In [ ]:
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_test, pred))
print("\n", classification_report(y_test, pred, digits=3))
print("ROC-AUC: %.3f" % roc_auc_score(y_test, proba))

### Step 5 · Which signals drive failures? (feature importance)

A model you can't explain is hard to trust on a factory floor. Random Forest exposes
**impurity-based importances** — how much each feature reduces prediction error
across the trees. This tells maintenance engineers *which* measurements to watch.

In [ ]:
imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
imp.plot.barh(ax=ax); ax.set_title("Random Forest feature importance")
plt.tight_layout(); plt.show()

### Step 6 · Explainability with SHAP (optional but powerful)

Feature *importance* tells you what matters globally; **SHAP** tells you *why a
specific prediction* was made — how each feature pushed this particular machine
toward "fail" or "healthy". This per-prediction explanation is exactly what a
maintenance work-order needs (and what the future `iiot-ai-rag` project will turn
into natural language).

> SHAP can be slow; we sample 300 test rows to keep it quick.

In [ ]:
try:
    import shap
    sample = X_test.sample(min(300, len(X_test)), random_state=0)
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(sample)
    sv_pos = sv[1] if isinstance(sv, list) else sv  # positive class
    shap.summary_plot(sv_pos, sample, show=True)
except Exception as e:
    print("SHAP skipped:", e)

## Part B — Remaining-Useful-Life regression (C-MAPSS, LSTM)

### Step 7 · Build sequences and standardise

We rebuild the feature pipeline from notebook `01` (RUL label → rolling features →
sequences), then **standardise** the inputs (zero mean, unit variance). Neural nets
train much better on standardised inputs. Crucially we **fit the scaler on training
data only** and reuse it on test data, to avoid leaking test-set statistics.

In [ ]:
cm = data.load_cmapss("FD001")
train_fe = features.add_rolling_features(features.add_rul(cm["train"], clip=125),
                                         features.feature_columns(cm["train"]))
cols = features.feature_columns(cm["train"])

X_seq, y_seq = features.make_sequences(train_fe, cols, seq_len=30)

scaler = utils.Standardizer().fit(X_seq.reshape(-1, X_seq.shape[-1]))
def scale(a):
    return scaler.transform(a.reshape(-1, a.shape[-1])).reshape(a.shape)
X_seq_s = scale(X_seq)
print("Sequence tensor:", X_seq_s.shape)

### Step 8 · Train the LSTM

An **LSTM** (Long Short-Term Memory network) is a recurrent neural net designed to
learn from sequences — it carries a memory across timesteps, so it can pick up on
*how* sensors are trending, not just their latest value. Our `LSTMRegressor`
(see `src/models.py`) is a 2-layer LSTM feeding a small dense head that outputs a
single number: predicted RUL.

We minimise **MSE** (mean squared error). Watch that both train and validation loss
fall and stay close — a big gap would signal overfitting.

> On Colab this uses the GPU automatically. On CPU, ~20 epochs takes a couple of
> minutes. Reduce `epochs` if you just want a quick look.

In [ ]:
model = models.LSTMRegressor(n_features=X_seq_s.shape[-1], hidden=64, layers=2)
history = models.train_lstm(model, X_seq_s, y_seq, epochs=20, batch_size=256)
utils.plot_loss(history, "LSTM training (MSE)"); plt.show()

### Step 9 · Predict RUL on the held-out test engines

The test set gives each engine's history truncated *before* failure; we take the
**last 30-cycle window** of each and predict its RUL, then compare to the ground
truth in `RUL_FD001.txt`.

We report two metrics:
- **RMSE** — average error in cycles (interpretable: "off by ~X cycles").
- **C-MAPSS score** — the official competition metric. It's **asymmetric**: it
  punishes *late* predictions (claiming more life than there is → unplanned failure)
  far more than early ones (conservative → safe). Lower is better. This asymmetry
  encodes the real-world cost: under-maintenance is worse than over-maintenance.

In [ ]:
X_test_seq = features.last_sequence_per_unit(cm["test"], cols, seq_len=30)
y_pred = models.predict_lstm(model, scale(X_test_seq))
y_true = cm["rul"]["rul"].to_numpy().clip(max=125)  # clip truth to match training

print("Test RMSE       : %.2f cycles" % utils.rmse(y_true, y_pred))
print("C-MAPSS score   : %.1f  (lower is better)" % utils.cmapss_score(y_true, y_pred))
utils.plot_rul_scatter(y_true, y_pred); plt.show()

### Step 10 · Reading the results

- Points **near the diagonal** = accurate predictions.
- Points **below** the line (predicted < true) = conservative → safe.
- Points **above** the line (predicted > true) = optimistic → risky; these drive up
  the C-MAPSS score the most.

A typical FD001 LSTM lands around **RMSE ≈ 15–20 cycles** — solid for a compact model
trained in minutes. Tuning (longer windows, more epochs, bidirectional LSTM) pushes
it lower.

**Next:** notebook `03` tackles the harder, more realistic case — detecting problems
**without any failure labels** (unsupervised anomaly detection).